# TacticsGPT Phase 1: Colab Training Notebook

This notebook builds a football tactics GPT in stages: corpus cleaning, tokenizer training, base pretraining, SFT LoRA fine-tuning, RL LoRA tuning, and generation/evaluation checks.

## Quick Run Map

1. Install dependencies.
2. Mount Google Drive and enter the project folder.
3. Add or clean the corpus only if `data/tactics_corpus.txt` is missing or intentionally being rebuilt.
4. Train the tokenizer only if `checkpoints/tokenizer/tokenizer.json` is missing or intentionally being rebuilt.
5. Run the stage you want to continue.

## Resume Disclaimer

Keep `START_FROM_SCRATCH = False` and keep `PROJECT_DIR` pointed at the same Google Drive folder when you want to continue from old weights.

- Pretraining resumes from the latest `checkpoints/pretrain/checkpoint_step_*.pt` unless `--no_resume` is used.
- SFT loads the pretrained base checkpoint and resumes the latest `checkpoints/sft/sft_step_*.pt`.
- RL loads the pretrained base plus the latest SFT adapter, then resumes the latest `rl_step_*.pt` in the chosen RL output folder.
- Evaluation cells only load existing weights and do not train.

Only set `START_FROM_SCRATCH = True`, change `PROJECT_DIR`, or change a training `--out_dir` when you intentionally want a fresh run.


## 0. Install Dependencies

Run once per Colab runtime. These packages are needed by the tokenizer, PyTorch model, progress bars, and data utilities.


In [ ]:
%pip install -q tokenizers tqdm numpy torch

## 1. Mount Drive And Project Folder

Run at the start of every Colab session. Keep `START_FROM_SCRATCH = False` to preserve existing Drive checkpoints and continue from old weights.


In [ ]:
from pathlib import Path
import os
import torch

from google.colab import drive

# Google Drive is always used so checkpoints survive Colab runtime resets.
drive.mount("/content/drive", force_remount=True)
PROJECT_DIR = Path("/content/drive/MyDrive/TacticsGPT_Phase1_Full_Pretrain")

# Keep this False for normal use. Because this notebook uses a new project folder,
# the first run starts clean, and later Colab restarts can resume from Drive.
START_FROM_SCRATCH = False

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

if START_FROM_SCRATCH:
    for target in [
        Path("checkpoints/pretrain"),
        Path("checkpoints/tokenizer"),
        Path("data/tactics_corpus.txt"),
    ]:
        if target.is_dir():
            import shutil
            shutil.rmtree(target)
        elif target.exists():
            target.unlink()
    print("Fresh run enabled: removed old generated tokenizer, cleaned corpus, and pretrain checkpoints.")
else:
    print("Resume mode: existing Drive corpus, tokenizer, and checkpoints will be kept.")

for folder in [
    "data/tokenizer",
    "checkpoints/pretrain",
    "checkpoints/tokenizer",
    "src",
]:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Enable Runtime > Change runtime type > GPU before full pretraining.")


## 2. Add And Clean The Training Corpus

Run this when `data/tactics_corpus.txt` is missing or when you intentionally want to rebuild the corpus from a new raw text dump.

Skip this while simply continuing old weights if your cleaned corpus is already in Drive.

The cleaned output is saved to `data/tactics_corpus.txt` as many training documents separated by blank lines. Long sections are chunked into smaller documents so the model does not treat a huge mixed tail as one single article.


In [ ]:
from pathlib import Path
import errno
import re
import unicodedata

raw_path = Path("data/raw_tactical_match_analysis.txt")
corpus_path = Path("data/tactics_corpus.txt")


def is_transport_disconnect(exc: OSError) -> bool:
    return (
        getattr(exc, "errno", None) == errno.ENOTCONN
        or "Transport endpoint is not connected" in str(exc)
    )


def raise_storage_help(path: Path, exc: OSError) -> None:
    raise RuntimeError(
        f"Colab cannot access {path!s}. This usually means the current project folder "
        "is on a disconnected Google Drive mount.\n\n"
        "Fix options:\n"
        "1. Remount Drive with: drive.mount('/content/drive', force_remount=True), then rerun from the setup cell.\n"
        "2. If the Runtime is stale, use Runtime > Restart session before rerunning."
    ) from exc


def safe_exists(path: Path) -> bool:
    try:
        return path.exists()
    except OSError as exc:
        if is_transport_disconnect(exc):
            raise_storage_help(path, exc)
        raise


def safe_size(path: Path) -> int:
    try:
        return path.stat().st_size
    except OSError as exc:
        if is_transport_disconnect(exc):
            raise_storage_help(path, exc)
        raise


def verify_data_storage() -> None:
    try:
        Path("data").mkdir(parents=True, exist_ok=True)
        probe = Path("data/.storage_probe")
        probe.write_text("ok", encoding="utf-8")
        probe.unlink(missing_ok=True)
    except OSError as exc:
        if is_transport_disconnect(exc):
            raise_storage_help(Path("data"), exc)
        raise


def maybe_fix_mojibake(text: str) -> str:
    markers = ("â€™", "â€œ", "â€", "â€”", "â€“", "Ã©")
    if sum(text.count(marker) for marker in markers) < 5:
        return text
    try:
        return text.encode("cp1252").decode("utf-8")
    except UnicodeError:
        return text


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", maybe_fix_mojibake(text))
    replacements = {
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u2013": "-",
        "\u2014": "-",
        "\u00a0": " ",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def is_wrapper_line(line: str) -> bool:
    stripped = line.strip()
    if not stripped:
        return False
    if stripped == "TACTICAL MATCH ANALYSIS ARTICLES":
        return True
    if stripped.startswith("Generated by "):
        return True
    if re.fullmatch(r"=+", stripped):
        return True
    if re.fullmatch(r"ARTICLE\s+#\d+", stripped, flags=re.IGNORECASE):
        return True
    return False


def is_document_boundary(line: str) -> bool:
    stripped = line.strip()
    return bool(re.fullmatch(r"ARTICLE\s+#\d+", stripped, flags=re.IGNORECASE))


def paragraph_chunks(paragraphs: list[str], target_chars: int = 9000, max_chars: int = 14000) -> list[str]:
    docs = []
    current = []
    current_len = 0

    for paragraph in paragraphs:
        paragraph = " ".join(paragraph.split())
        if not paragraph:
            continue

        if current and current_len + len(paragraph) > target_chars:
            docs.append("\n\n".join(current))
            current = []
            current_len = 0

        current.append(paragraph)
        current_len += len(paragraph)

        if current_len >= max_chars:
            docs.append("\n\n".join(current))
            current = []
            current_len = 0

    if current:
        docs.append("\n\n".join(current))

    return docs


def extract_training_documents(raw_text: str, min_chars: int = 400) -> list[str]:
    text = normalize_text(raw_text).replace("\r\n", "\n").replace("\r", "\n")
    sections: list[list[str]] = []
    current_lines: list[str] = []

    for line in text.splitlines():
        if is_document_boundary(line) and current_lines:
            sections.append(current_lines)
            current_lines = []
            continue
        if is_wrapper_line(line):
            continue
        current_lines.append(line.rstrip())

    if current_lines:
        sections.append(current_lines)

    documents = []
    for section in sections:
        section_text = "\n".join(section).strip()
        if not section_text:
            continue
        paragraphs = [part.strip() for part in re.split(r"\n\s*\n+", section_text) if part.strip()]
        for doc in paragraph_chunks(paragraphs):
            if len(doc) >= min_chars:
                documents.append(doc)

    return documents


def build_clean_corpus(raw_file: Path, clean_file: Path) -> list[str]:
    raw_text = raw_file.read_text(encoding="utf-8", errors="replace")
    documents = extract_training_documents(raw_text)
    if not documents:
        raise ValueError("No training documents were found. Check the uploaded text file format.")
    clean_file.write_text("\n\n\n".join(documents) + "\n", encoding="utf-8")
    return documents


verify_data_storage()
raw_exists = safe_exists(raw_path)
corpus_exists = safe_exists(corpus_path)

if not raw_exists and not corpus_exists:
    try:
        from google.colab import files

        print("Upload tactical_match_analysis_5mb.txt or a similar tactics .txt dump.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")

        first_name = next(iter(uploaded))
        raw_path.write_bytes(uploaded[first_name])
        raw_exists = True
        print(f"Saved uploaded file as {raw_path}")
    except Exception as exc:
        print("Could not upload automatically:", exc)
        print("Place your raw text dump at data/raw_tactical_match_analysis.txt before continuing.")

if raw_exists:
    documents = build_clean_corpus(raw_path, corpus_path)
    print(f"Cleaned {len(documents):,} training documents/sections into {corpus_path}")
elif corpus_exists and safe_size(corpus_path) > 0:
    documents = [part.strip() for part in re.split(r"\n{3,}", corpus_path.read_text(encoding="utf-8", errors="replace")) if part.strip()]
    print(f"Using existing cleaned corpus with {len(documents):,} training documents/sections.")
else:
    raise FileNotFoundError("Missing data/raw_tactical_match_analysis.txt or data/tactics_corpus.txt")

print(f"Clean corpus size: {safe_size(corpus_path):,} bytes")
print("\nFirst cleaned document preview:\n")
print(documents[0][:1200])

## 3. Write Project Source Files

These cells create the Python scripts used by the notebook. Run them after setup in a fresh Colab session, or after editing any script logic.


### 3.1 Shared Helpers - `src/utils.py`

Creates device selection, seed setup, folder creation, and checkpoint helper functions used by the training scripts.


In [ ]:
%%writefile src/utils.py
import glob
import os
import random
import re
from pathlib import Path

import numpy as np
import torch


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"


def ensure_dirs(*paths: str) -> None:
    for path in paths:
        Path(path).mkdir(parents=True, exist_ok=True)


def checkpoint_step(path: str) -> int:
    match = re.search(r"checkpoint_step_(\d+)\.pt$", os.path.basename(path))
    return int(match.group(1)) if match else -1


def latest_checkpoint(out_dir: str) -> str | None:
    checkpoints = glob.glob(os.path.join(out_dir, "checkpoint_step_*.pt"))
    if not checkpoints:
        return None
    return max(checkpoints, key=checkpoint_step)


def save_checkpoint(
    path: str,
    model,
    optimizer,
    step: int,
    epoch: int,
    tokenizer_path: str,
    config: dict,
    best_val_loss: float | None = None,
) -> None:
    payload = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "step": step,
        "epoch": epoch,
        "tokenizer_path": tokenizer_path,
        "config": config,
    }
    if best_val_loss is not None:
        payload["best_val_loss"] = best_val_loss
    torch.save(payload, path)

### 3.2 Corpus Builder - `src/build_corpus.py`

Writes the reusable corpus-cleaning script for raw tactical text dumps.


In [ ]:
%%writefile src/build_corpus.py
import argparse
from pathlib import Path
import re
import unicodedata


def maybe_fix_mojibake(text: str) -> str:
    markers = ("â€™", "â€œ", "â€", "â€”", "â€“", "Ã©")
    if sum(text.count(marker) for marker in markers) < 5:
        return text
    try:
        return text.encode("cp1252").decode("utf-8")
    except UnicodeError:
        return text


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", maybe_fix_mojibake(text))
    replacements = {
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u2013": "-",
        "\u2014": "-",
        "\u00a0": " ",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def is_wrapper_line(line: str) -> bool:
    stripped = line.strip()
    if not stripped:
        return False
    if stripped == "TACTICAL MATCH ANALYSIS ARTICLES":
        return True
    if stripped.startswith("Generated by "):
        return True
    if re.fullmatch(r"=+", stripped):
        return True
    if re.fullmatch(r"ARTICLE\s+#\d+", stripped, flags=re.IGNORECASE):
        return True
    return False


def is_document_boundary(line: str) -> bool:
    stripped = line.strip()
    return bool(re.fullmatch(r"ARTICLE\s+#\d+", stripped, flags=re.IGNORECASE))


def paragraph_chunks(paragraphs: list[str], target_chars: int = 9000, max_chars: int = 14000) -> list[str]:
    docs = []
    current = []
    current_len = 0

    for paragraph in paragraphs:
        paragraph = " ".join(paragraph.split())
        if not paragraph:
            continue

        if current and current_len + len(paragraph) > target_chars:
            docs.append("\n\n".join(current))
            current = []
            current_len = 0

        current.append(paragraph)
        current_len += len(paragraph)

        if current_len >= max_chars:
            docs.append("\n\n".join(current))
            current = []
            current_len = 0

    if current:
        docs.append("\n\n".join(current))

    return docs


def extract_training_documents(raw_text: str, min_chars: int = 400) -> list[str]:
    text = normalize_text(raw_text).replace("\r\n", "\n").replace("\r", "\n")
    sections: list[list[str]] = []
    current_lines: list[str] = []

    for line in text.splitlines():
        if is_document_boundary(line) and current_lines:
            sections.append(current_lines)
            current_lines = []
            continue
        if is_wrapper_line(line):
            continue
        current_lines.append(line.rstrip())

    if current_lines:
        sections.append(current_lines)

    documents = []
    for section in sections:
        section_text = "\n".join(section).strip()
        if not section_text:
            continue
        paragraphs = [part.strip() for part in re.split(r"\n\s*\n+", section_text) if part.strip()]
        for doc in paragraph_chunks(paragraphs):
            if len(doc) >= min_chars:
                documents.append(doc)

    return documents


def build_clean_corpus(raw_file: str, clean_file: str, min_chars: int = 400) -> None:
    raw_path = Path(raw_file)
    out_path = Path(clean_file)
    if not raw_path.exists() or raw_path.stat().st_size == 0:
        raise FileNotFoundError(f"Raw corpus not found or empty: {raw_path}")

    documents = extract_training_documents(raw_path.read_text(encoding="utf-8", errors="replace"), min_chars)
    if not documents:
        raise ValueError("No training documents were found. Check the source text format.")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text("\n\n\n".join(documents) + "\n", encoding="utf-8")
    print(f"Wrote {len(documents):,} cleaned training documents/sections to {out_path}")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--raw", default="data/raw_tactical_match_analysis.txt")
    parser.add_argument("--out", default="data/tactics_corpus.txt")
    parser.add_argument("--min_chars", type=int, default=400)
    args = parser.parse_args()
    build_clean_corpus(args.raw, args.out, args.min_chars)


if __name__ == "__main__":
    main()

### 3.3 Tokenizer Trainer - `src/tokenizer_train.py`

Writes the BPE tokenizer training script.


In [ ]:
%%writefile src/tokenizer_train.py
import argparse
from pathlib import Path

from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.models import BPE
from tokenizers.normalizers import Lowercase, NFKC, Sequence
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.processors import TemplateProcessing
from tokenizers.trainers import BpeTrainer


SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>"]


def train_tokenizer(corpus_path: str, out_dir: str, vocab_size: int = 8000, lowercase: bool = False) -> None:
    corpus = Path(corpus_path)
    if not corpus.exists() or corpus.stat().st_size == 0:
        raise FileNotFoundError(f"Corpus not found or empty: {corpus}")

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    tokenizer = Tokenizer(BPE(unk_token="<unk>"))
    normalizers = [NFKC()]
    if lowercase:
        normalizers.append(Lowercase())
    tokenizer.normalizer = Sequence(normalizers)
    tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=True)
    tokenizer.decoder = ByteLevelDecoder()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
        show_progress=True,
    )
    tokenizer.train([str(corpus)], trainer)

    bos_id = tokenizer.token_to_id("<bos>")
    eos_id = tokenizer.token_to_id("<eos>")
    tokenizer.post_processor = TemplateProcessing(
        single="<bos> $A <eos>",
        special_tokens=[("<bos>", bos_id), ("<eos>", eos_id)],
    )

    tokenizer.save(str(out / "tokenizer.json"))
    tokenizer.model.save(str(out))

    print("Saved tokenizer files:")
    print(" -", out / "tokenizer.json")
    print(" -", out / "vocab.json")
    print(" -", out / "merges.txt")
    print("Actual vocab size:", tokenizer.get_vocab_size())


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--corpus", default="data/tactics_corpus.txt")
    parser.add_argument("--out_dir", default="checkpoints/tokenizer")
    parser.add_argument("--vocab_size", type=int, default=8000)
    parser.add_argument("--lowercase", action="store_true")
    args = parser.parse_args()
    train_tokenizer(args.corpus, args.out_dir, args.vocab_size, args.lowercase)


if __name__ == "__main__":
    main()

### 3.4 Pretraining Dataset - `src/dataset.py`

Writes the article-aware next-token dataset used during base pretraining.


In [ ]:
%%writefile src/dataset.py
import re
from pathlib import Path

import torch
from torch.utils.data import Dataset
from tokenizers import Tokenizer


def split_article_documents(text: str) -> list[str]:
    docs = [part.strip() for part in re.split(r"\n{3,}", text) if part.strip()]
    return docs if docs else [text.strip()]


class TacticsDataset(Dataset):
    def __init__(
        self,
        corpus_path: str,
        tokenizer_path: str,
        context_length: int = 256,
        stride: int | None = None,
    ) -> None:
        corpus = Path(corpus_path)
        if not corpus.exists() or corpus.stat().st_size == 0:
            raise FileNotFoundError(f"Corpus not found or empty: {corpus}")

        self.context_length = context_length
        self.stride = stride or context_length
        self.tokenizer = Tokenizer.from_file(tokenizer_path)
        self.documents: list[list[int]] = []
        self.examples: list[tuple[int, int]] = []

        text = corpus.read_text(encoding="utf-8", errors="ignore")
        for document in split_article_documents(text):
            token_ids = self.tokenizer.encode(document).ids
            if len(token_ids) < context_length + 1:
                continue
            doc_index = len(self.documents)
            self.documents.append(token_ids)
            self.examples.extend(
                (doc_index, start)
                for start in range(0, len(token_ids) - context_length, self.stride)
            )

        self.num_documents = len(self.documents)
        self.num_tokens = sum(len(doc) for doc in self.documents)
        if not self.examples:
            raise ValueError(
                "No training windows were created. "
                f"Try reducing context_length below {context_length}, "
                "or check that data/tactics_corpus.txt contains full article bodies."
            )

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, idx: int):
        doc_index, start = self.examples[idx]
        tokens = self.documents[doc_index]
        end = start + self.context_length
        x = torch.tensor(tokens[start:end], dtype=torch.long)
        y = torch.tensor(tokens[start + 1 : end + 1], dtype=torch.long)
        return x, y

### 3.5 GPT Model - `src/model.py`

Writes the decoder-only GPT architecture used by pretraining, SFT, RL, and generation.


In [ ]:
%%writefile src/model.py
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class GPTConfig:
    vocab_size: int = 8000
    context_length: int = 256
    n_layers: int = 4
    n_heads: int = 4
    d_model: int = 256
    ffn_hidden: int = 1024
    dropout: float = 0.1


class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        assert config.d_model % config.n_heads == 0
        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads

        self.qkv = nn.Linear(config.d_model, 3 * config.d_model)
        self.out = nn.Linear(config.d_model, config.d_model)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        mask = torch.tril(torch.ones(config.context_length, config.context_length))
        self.register_buffer("causal_mask", mask.view(1, 1, config.context_length, config.context_length))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq_len, channels = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(channels, dim=2)

        q = q.view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        scores = scores.masked_fill(self.causal_mask[:, :, :seq_len, :seq_len] == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)

        y = weights @ v
        y = y.transpose(1, 2).contiguous().view(batch, seq_len, channels)
        return self.resid_dropout(self.out(y))


class FeedForward(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.d_model, config.ffn_hidden),
            nn.GELU(),
            nn.Linear(config.ffn_hidden, config.d_model),
            nn.Dropout(config.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Block(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        self.ln1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.d_model)
        self.ffn = FeedForward(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.position_embedding = nn.Embedding(config.context_length, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layers)])
        self.ln_f = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        self.token_embedding.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        batch, seq_len = idx.shape
        if seq_len > self.config.context_length:
            raise ValueError(f"Sequence length {seq_len} exceeds context length {self.config.context_length}")

        positions = torch.arange(0, seq_len, dtype=torch.long, device=idx.device).unsqueeze(0)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        x = self.dropout(x)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int = 80,
        temperature: float = 0.8,
        top_k: int | None = 50,
        greedy: bool = False,
    ) -> torch.Tensor:
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.context_length :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]

            if greedy:
                next_id = torch.argmax(logits, dim=-1, keepdim=True)
            else:
                logits = logits / max(temperature, 1e-6)
                if top_k is not None and top_k > 0:
                    values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < values[:, [-1]]] = float("-inf")
                probs = F.softmax(logits, dim=-1)
                next_id = torch.multinomial(probs, num_samples=1)

            idx = torch.cat([idx, next_id], dim=1)
        return idx

### 3.6 Pretraining Script - `src/train_pretrain.py`

Writes the base training loop. It saves step checkpoints, a best validation checkpoint, and resumes automatically unless `--no_resume` is used.


In [ ]:
%%writefile src/train_pretrain.py
import argparse
import math
from dataclasses import asdict
from pathlib import Path

import torch
from tokenizers import Tokenizer
from torch.utils.data import DataLoader, Subset, random_split
from tqdm.auto import tqdm

from dataset import TacticsDataset
from model import GPT, GPTConfig
from utils import ensure_dirs, get_device, latest_checkpoint, save_checkpoint, set_seed


def make_loaders(dataset, batch_size: int, val_fraction: float, num_workers: int, seed: int, device: str):
    val_size = max(1, int(len(dataset) * val_fraction)) if val_fraction > 0 and len(dataset) > 20 else 0
    train_size = len(dataset) - val_size
    generator = torch.Generator().manual_seed(seed)

    if val_size:
        train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)
    else:
        train_dataset = dataset
        val_dataset = None

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
        pin_memory=(device == "cuda"),
    )
    val_loader = None
    if val_dataset is not None:
        val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            num_workers=num_workers,
            pin_memory=(device == "cuda"),
        )
    return train_loader, val_loader, train_size, val_size


def learning_rate(step: int, base_lr: float, warmup_steps: int, max_steps: int) -> float:
    if warmup_steps > 0 and step < warmup_steps:
        return base_lr * (step + 1) / warmup_steps
    if max_steps <= warmup_steps:
        return base_lr
    progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
    progress = min(1.0, max(0.0, progress))
    return 0.1 * base_lr + 0.9 * base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))


@torch.no_grad()
def evaluate(model, loader, device: str, max_batches: int = 50) -> float | None:
    if loader is None:
        return None
    model.eval()
    losses = []
    for batch_idx, (x, y) in enumerate(loader):
        if batch_idx >= max_batches:
            break
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses) / max(1, len(losses))


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--corpus", default="data/tactics_corpus.txt")
    parser.add_argument("--tokenizer", default="checkpoints/tokenizer/tokenizer.json")
    parser.add_argument("--out_dir", default="checkpoints/pretrain")
    parser.add_argument("--context_length", type=int, default=256)
    parser.add_argument("--n_layers", type=int, default=4)
    parser.add_argument("--n_heads", type=int, default=4)
    parser.add_argument("--d_model", type=int, default=256)
    parser.add_argument("--ffn_hidden", type=int, default=1024)
    parser.add_argument("--dropout", type=float, default=0.1)
    parser.add_argument("--batch_size", type=int, default=16)
    parser.add_argument("--grad_accum_steps", type=int, default=2)
    parser.add_argument("--epochs", type=int, default=30)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--warmup_steps", type=int, default=500)
    parser.add_argument("--stride", type=int, default=128)
    parser.add_argument("--save_every", type=int, default=250)
    parser.add_argument("--eval_every", type=int, default=250)
    parser.add_argument("--val_fraction", type=float, default=0.05)
    parser.add_argument("--eval_batches", type=int, default=50)
    parser.add_argument("--max_steps", type=int, default=0, help="0 means train for all epochs")
    parser.add_argument("--num_workers", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--no_resume", action="store_true")
    args = parser.parse_args()

    set_seed(args.seed)
    ensure_dirs(args.out_dir)
    device = get_device()
    use_amp = device == "cuda"
    print("Device:", device)
    print("AMP:", use_amp)

    tokenizer = Tokenizer.from_file(args.tokenizer)
    config = GPTConfig(
        vocab_size=tokenizer.get_vocab_size(),
        context_length=args.context_length,
        n_layers=args.n_layers,
        n_heads=args.n_heads,
        d_model=args.d_model,
        ffn_hidden=args.ffn_hidden,
        dropout=args.dropout,
    )

    dataset = TacticsDataset(args.corpus, args.tokenizer, args.context_length, args.stride)
    train_loader, val_loader, train_size, val_size = make_loaders(
        dataset, args.batch_size, args.val_fraction, args.num_workers, args.seed, device
    )
    steps_per_epoch = max(1, len(train_loader) // max(1, args.grad_accum_steps))
    planned_steps = args.max_steps if args.max_steps else steps_per_epoch * args.epochs

    print(f"Dataset training documents/sections: {dataset.num_documents:,}")
    print(f"Dataset tokens: {dataset.num_tokens:,}")
    print(f"Dataset windows: {len(dataset):,}")
    print(f"Train windows: {train_size:,}")
    print(f"Validation windows: {val_size:,}")
    print(f"Optimization steps planned: {planned_steps:,}")
    print(f"Effective batch size: {args.batch_size * args.grad_accum_steps:,}")

    model = GPT(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, betas=(0.9, 0.95), weight_decay=0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    global_step = 0
    start_epoch = 0
    best_val_loss = float("inf")

    ckpt = None if args.no_resume else latest_checkpoint(args.out_dir)
    if ckpt:
        print("Resuming from:", ckpt)
        state = torch.load(ckpt, map_location=device)
        model.load_state_dict(state["model_state_dict"])
        optimizer.load_state_dict(state["optimizer_state_dict"])
        global_step = int(state.get("step", 0))
        start_epoch = int(state.get("epoch", 0))
        best_val_loss = float(state.get("best_val_loss", best_val_loss))

    model.train()
    optimizer.zero_grad(set_to_none=True)

    for epoch in range(start_epoch, args.epochs):
        progress = tqdm(train_loader, desc=f"epoch {epoch + 1}/{args.epochs}")
        for micro_step, (x, y) in enumerate(progress, start=1):
            lr = learning_rate(global_step, args.lr, args.warmup_steps, planned_steps)
            for group in optimizer.param_groups:
                group["lr"] = lr

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                _, loss = model(x, y)
                loss_to_backprop = loss / args.grad_accum_steps

            scaler.scale(loss_to_backprop).backward()

            if micro_step % args.grad_accum_steps != 0:
                progress.set_postfix(loss=f"{loss.item():.4f}", step=global_step, lr=f"{lr:.2e}")
                continue

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1
            progress.set_postfix(loss=f"{loss.item():.4f}", step=global_step, lr=f"{lr:.2e}")

            should_eval = val_loader is not None and args.eval_every > 0 and global_step % args.eval_every == 0
            if should_eval:
                val_loss = evaluate(model, val_loader, device, args.eval_batches)
                print(f"\\nValidation loss at step {global_step}: {val_loss:.4f}")
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_path = Path(args.out_dir) / "checkpoint_best.pt"
                    save_checkpoint(str(best_path), model, optimizer, global_step, epoch, args.tokenizer, asdict(config), best_val_loss)
                    print("Saved new best checkpoint:", best_path)

            if global_step % args.save_every == 0:
                path = Path(args.out_dir) / f"checkpoint_step_{global_step}.pt"
                save_checkpoint(str(path), model, optimizer, global_step, epoch, args.tokenizer, asdict(config), best_val_loss)
                print("\\nSaved", path)

            if args.max_steps and global_step >= args.max_steps:
                final_path = Path(args.out_dir) / f"checkpoint_step_{global_step}.pt"
                save_checkpoint(str(final_path), model, optimizer, global_step, epoch, args.tokenizer, asdict(config), best_val_loss)
                print("\\nReached max_steps. Saved", final_path)
                return

        # Save at the end of every epoch so progress survives Colab runtime resets.
        epoch_path = Path(args.out_dir) / f"checkpoint_step_{global_step}.pt"
        save_checkpoint(str(epoch_path), model, optimizer, global_step, epoch + 1, args.tokenizer, asdict(config), best_val_loss)
        print("\\nSaved end-of-epoch checkpoint:", epoch_path)

    final_path = Path(args.out_dir) / f"checkpoint_step_{global_step}.pt"
    save_checkpoint(str(final_path), model, optimizer, global_step, args.epochs, args.tokenizer, asdict(config), best_val_loss)
    print("Training complete. Saved", final_path)
    if best_val_loss < float("inf"):
        print(f"Best validation loss: {best_val_loss:.4f}")


if __name__ == "__main__":
    main()

### 3.7 Generation Script - `src/generate.py`

Writes the base-model text generation utility.


In [ ]:
%%writefile src/generate.py
import argparse
from pathlib import Path

import torch
from tokenizers import Tokenizer

from model import GPT, GPTConfig
from utils import get_device, latest_checkpoint


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--prompt", required=True)
    parser.add_argument("--checkpoint", default="checkpoints/pretrain/checkpoint_best.pt")
    parser.add_argument("--checkpoint_dir", default="checkpoints/pretrain")
    parser.add_argument("--tokenizer", default="checkpoints/tokenizer/tokenizer.json")
    parser.add_argument("--max_new_tokens", type=int, default=100)
    parser.add_argument("--temperature", type=float, default=0.8)
    parser.add_argument("--top_k", type=int, default=50)
    parser.add_argument("--greedy", action="store_true")
    args = parser.parse_args()

    checkpoint = args.checkpoint
    if not checkpoint or not Path(checkpoint).exists():
        checkpoint = latest_checkpoint(args.checkpoint_dir)
    if not checkpoint:
        raise FileNotFoundError(f"No checkpoint found in {args.checkpoint_dir}")

    device = get_device()
    state = torch.load(checkpoint, map_location=device)
    config = GPTConfig(**state["config"])

    tokenizer_path = args.tokenizer or state.get("tokenizer_path")
    tokenizer = Tokenizer.from_file(tokenizer_path)

    model = GPT(config).to(device)
    model.load_state_dict(state["model_state_dict"])
    model.eval()

    input_ids = tokenizer.encode(args.prompt).ids
    idx = torch.tensor([input_ids], dtype=torch.long, device=device)

    output = model.generate(
        idx,
        max_new_tokens=args.max_new_tokens,
        temperature=args.temperature,
        top_k=args.top_k,
        greedy=args.greedy,
    )
    text = tokenizer.decode(output[0].tolist(), skip_special_tokens=True)

    print("Checkpoint:", checkpoint)
    if "best_val_loss" in state:
        print("Best validation loss:", state["best_val_loss"])
    print("\nGenerated text:\n")
    print(text)


if __name__ == "__main__":
    main()

### 3.8 Pretraining README

Writes a short project README that mirrors the base pretraining workflow.


In [ ]:
%%writefile README_01_PRETRAIN.md
# TacticsGPT Phase 1: Pretraining

This project trains a small decoder-only GPT model on football tactical text using next-token prediction.

## Data

The provided Gemini article dump format is supported. Put the raw file at:

```text
data/raw_tactical_match_analysis.txt
```

Then build the cleaned corpus:

```bash
python src/build_corpus.py --raw data/raw_tactical_match_analysis.txt --out data/tactics_corpus.txt
```

This removes `ARTICLE #n` wrappers and separator bars while keeping match analysis, coach speeches, explanations, and tactical notes as training documents.

The cleaned text is written at:

```text
data/tactics_corpus.txt
```

Do not use JSON, CSV, labels, or question-answer pairs for Phase 1.

## Tokenizer

```bash
python src/tokenizer_train.py --corpus data/tactics_corpus.txt --out_dir checkpoints/tokenizer --vocab_size 8000
```

Outputs:

- `checkpoints/tokenizer/tokenizer.json`
- `checkpoints/tokenizer/vocab.json`
- `checkpoints/tokenizer/merges.txt`

## Pretraining

```bash
python src/train_pretrain.py --corpus data/tactics_corpus.txt --tokenizer checkpoints/tokenizer/tokenizer.json --epochs 30 --grad_accum_steps 2
```

The trainer automatically resumes from the latest `checkpoints/pretrain/checkpoint_step_*.pt` unless `--no_resume` is passed.

## Generation

```bash
python src/generate.py --prompt "How should a 4-4-2 defend centrally?"
```

This phase only trains tactical language modeling. It is not a RAG assistant and does not aim for factual perfection yet.

## 4. Train The Tokenizer

This step requires the cleaned corpus at `data/tactics_corpus.txt`.

Resume note: existing model checkpoints expect the same tokenizer they were trained with. If you are continuing old weights and `checkpoints/tokenizer/tokenizer.json` already exists, skip this cell unless you intentionally want to rebuild the tokenizer and retrain compatible weights.

If this cell fails, do not run training yet. Go back to **Add And Clean The Training Corpus**, upload the raw `.txt` file, and confirm it prints the cleaned document count.


In [ ]:
from pathlib import Path

corpus_file = Path("data/tactics_corpus.txt")
if not corpus_file.exists() or corpus_file.stat().st_size == 0:
    raise FileNotFoundError(
        "Missing data/tactics_corpus.txt.\n"
        "Run the 'Add The Training Corpus' cell first and upload tactical_match_analysis_5mb.txt.\n"
        "Do not use src/build_corpus.txt as the corpus path; src/build_corpus.py is only the cleaning script."
    )

print(f"Tokenizer input corpus: {corpus_file} ({corpus_file.stat().st_size:,} bytes)")
print("Tokenizer note: skip this when continuing old weights and tokenizer.json already exists.")
!python src/tokenizer_train.py --corpus data/tactics_corpus.txt --out_dir checkpoints/tokenizer --vocab_size 8000


## 5. Pretrain Or Continue The Base GPT

Run this only after tokenizer training succeeds and creates `checkpoints/tokenizer/tokenizer.json`.

Resume Disclaimer: this training script automatically loads the latest checkpoint in `checkpoints/pretrain`. It starts from zero only when no checkpoint exists or when `--no_resume` is added.

The run below is a full pretrain setup:

- `30` epochs
- no artificial `max_steps` cutoff
- checkpoint every `250` steps
- validation loss every `250` steps
- best checkpoint saved separately
- AMP enabled on CUDA
- gradient accumulation for a larger effective batch

If Colab runs out of GPU memory, reduce `--batch_size` to `8`, then `4`. Keep `--grad_accum_steps` at `2` or increase it to preserve the effective batch size.


In [ ]:
from pathlib import Path

corpus_file = Path("data/tactics_corpus.txt")
tokenizer_file = Path("checkpoints/tokenizer/tokenizer.json")

if not corpus_file.exists() or corpus_file.stat().st_size == 0:
    raise FileNotFoundError("Missing data/tactics_corpus.txt. Run the corpus upload/cleaning cell first.")
if not tokenizer_file.exists() or tokenizer_file.stat().st_size == 0:
    raise FileNotFoundError("Missing checkpoints/tokenizer/tokenizer.json. Run the tokenizer training cell first.")

print(f"Training corpus: {corpus_file} ({corpus_file.stat().st_size:,} bytes)")
print(f"Tokenizer: {tokenizer_file}")
print("Resume Disclaimer: pretraining loads the latest checkpoints/pretrain/checkpoint_step_*.pt unless --no_resume is added.")

!python src/train_pretrain.py \
  --corpus data/tactics_corpus.txt \
  --tokenizer checkpoints/tokenizer/tokenizer.json \
  --epochs 30 \
  --batch_size 16 \
  --grad_accum_steps 2 \
  --context_length 256 \
  --stride 128 \
  --lr 3e-4 \
  --warmup_steps 500 \
  --save_every 250 \
  --eval_every 250 \
  --val_fraction 0.05 \
  --max_steps 0


## 6. Generate From The Base Model

Run this only after pretraining has saved at least one checkpoint in `checkpoints/pretrain`. This cell loads existing weights for inference only.


In [ ]:
from pathlib import Path

checkpoint_dir = Path("checkpoints/pretrain")
has_best = (checkpoint_dir / "checkpoint_best.pt").exists()
has_step = any(checkpoint_dir.glob("checkpoint_step_*.pt"))

if not has_best and not has_step:
    raise FileNotFoundError("No training checkpoint found. Run pretraining first and wait until it saves a checkpoint.")

!python src/generate.py \
  --prompt "How should a 4-4-2 defend centrally?" \
  --max_new_tokens 120 \
  --temperature 0.8 \
  --top_k 50

## 6.1 Optional Commands

Use these as small reference snippets for generation modes, intentional fresh starts, or larger model experiments.


In [ ]:
# Greedy decoding:
# !python src/generate.py --prompt "The midfield pivot must" --greedy --max_new_tokens 80

# Resume training happens automatically. To force a fresh run:
# !python src/train_pretrain.py --no_resume --max_steps 1000

# Larger model example, if your GPU has enough memory:
# !python src/train_pretrain.py --d_model 384 --n_heads 6 --n_layers 6 --batch_size 8